# 02 · Baseline recommender demo

Runs `ml/baseline_recommender.py` on the real O*NET 31.0 data for two hand-made test profiles.
These are sanity tests, not accuracy results (there is no student data yet).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("..").resolve()))   # lets the notebook find the ml folder
from ml.baseline_recommender import load_onet, recommend, explain

ONET_DIR = Path("..") / "data" / "raw" / "onet"
data = load_onet(ONET_DIR)

print("Occupations in O*NET    :", len(data["occupations"]))
print("Occupations we can rank :", len(data["codes"]))
print("Skills used             :", data["skill_importance"].shape[1])
print("Knowledge areas used    :", data["knowledge_importance"].shape[1])
print("Job zones loaded        :", "job_zone" in data)

In [ ]:
# Test profile A: biology and research (ratings 1-5, typed by hand)
student_a = {
    "interests": {"Realistic": 2.8, "Investigative": 4.6, "Artistic": 3.4,
                  "Social": 3.2, "Enterprising": 2.1, "Conventional": 3.0},
    "skills": {"Mathematics": 3, "Programming": 3, "Problem Solving": 4, "Logical Reasoning": 4,
               "Critical Thinking": 4, "Research": 5, "Communication": 3, "Writing": 4,
               "Public Speaking": 2, "Leadership": 2, "Teamwork": 3, "Time Management": 3,
               "Project Management": 2, "Financial/Business Skills": 1, "Teaching": 3,
               "Technical/Practical Skills": 3},
    "subjects": {"Mathematics": 3, "Physics": 3, "Chemistry": 4, "Biology": 5,
                 "Computer Science": 4, "Economics": 2, "Commerce/Accounting": 1,
                 "Languages": 3, "Social Sciences": 2, "Arts/Design": 3},
}

In [ ]:
recs = recommend(student_a, data, top_n=5, min_job_zone=3)
recs

In [ ]:
# Why each career, and which skills to build
for _, row in recs.iterrows():
    reasons, gaps = explain(student_a, data, row["code"])
    print(f"\n#{row['rank']}  {row['career']}  -  {row['match_score']}/100  ({row['domain']})")
    print("  Why this career?")
    for r in reasons or ["(overall profile similarity)"]:
        print("   +", r)
    print("  Skills to build:")
    print(gaps.to_string(index=False) if len(gaps) else "   none")

In [ ]:
# Test profile B: arts and communication
student_b = {
    "interests": {"Realistic": 1.8, "Investigative": 2.5, "Artistic": 4.4,
                  "Social": 3.9, "Enterprising": 4.2, "Conventional": 2.6},
    "skills": {"Mathematics": 2, "Programming": 1, "Problem Solving": 3, "Logical Reasoning": 3,
               "Critical Thinking": 3, "Research": 3, "Communication": 5, "Writing": 4,
               "Public Speaking": 4, "Leadership": 4, "Teamwork": 4, "Time Management": 3,
               "Project Management": 3, "Financial/Business Skills": 3, "Teaching": 3,
               "Technical/Practical Skills": 1},
    "subjects": {"Mathematics": 2, "Physics": 1, "Chemistry": 1, "Biology": 2,
                 "Computer Science": 2, "Economics": 3, "Commerce/Accounting": 3,
                 "Languages": 5, "Social Sciences": 4, "Arts/Design": 5},
}
recommend(student_b, data, top_n=5, min_job_zone=3)

In [ ]:
# Careers the system can rank, by field (saved for the report and slides)
import matplotlib.pyplot as plt
from ml.baseline_recommender import SOC_MAJOR_GROUPS

domains = (pd.Series([SOC_MAJOR_GROUPS.get(c[:2], "Other") for c in data["codes"]])
             .value_counts().sort_values())

ax = domains.plot.barh(figsize=(8, 8), color="#2B3FD1")
ax.set_title("Careers our system can rank, by domain (O*NET 31.0)")
ax.set_xlabel("Number of occupations")
plt.tight_layout()
plt.savefig("../docs/careers_by_domain.png", dpi=200)
plt.show()

In [ ]:
# Export all rankable careers to docs/careers.json for the prototype (Data tab)
import json
from ml.baseline_recommender import RIASEC, SOC_MAJOR_GROUPS

codes = list(data["codes"])
skills = list(data["skill_importance"].columns)
knowledge = list(data["knowledge_importance"].columns)
interests = data["interests"].reindex(index=codes, columns=RIASEC)
skill_im = data["skill_importance"].reindex(index=codes, columns=skills)
skill_lv = data["skill_level"].reindex(index=codes, columns=skills)
know_im = data["knowledge_importance"].reindex(index=codes, columns=knowledge)
job_zone = data.get("job_zone", pd.Series(dtype=float)).reindex(codes)

def clean(values):
    return [None if pd.isna(v) else round(float(v), 2) for v in values]

export = {"source": "O*NET 31.0", "riasec": RIASEC, "skills": skills,
          "knowledge": knowledge, "careers": []}
for c in codes:
    export["careers"].append({
        "code": c,
        "title": data["occupations"].loc[c, "Title"],
        "description": data["occupations"].loc[c, "Description"],
        "domain": SOC_MAJOR_GROUPS.get(c[:2], "Other"),
        "job_zone": None if pd.isna(job_zone[c]) else int(job_zone[c]),
        "interests": clean(interests.loc[c]),
        "skill_im": clean(skill_im.loc[c]),
        "skill_lv": clean(skill_lv.loc[c]),
        "know_im": clean(know_im.loc[c]),
    })

out_file = Path("..") / "docs" / "careers.json"
out_file.write_text(json.dumps(export), encoding="utf-8")
print(len(export["careers"]), "careers exported to", out_file.resolve())
print("File size:", round(out_file.stat().st_size / 1024), "KB")